# split pdbs into models

In [4]:
import os

def split_pdb_by_model(input_pdb, output_folder="output", name_schema="model_{model_num}"):
    """
    Split a multi-model PDB file into separate files, one per model.
    
    Parameters:
    -----------
    input_pdb : str
        Path to the input PDB file containing multiple models
    output_folder : str
        Folder where output PDB files will be saved (default: "output")
    name_schema : str
        Naming schema for output PDB files (default: "model_{model_num}")
        
    Returns:
    --------
    list
        List of output filenames created
    """
    # Read PDB file
    with open(input_pdb, 'r') as f:
        pdb_lines = f.readlines()
    
    output_files = []
    current_model = None
    current_lines = []
    model_lines = {}
    
    # Process the file line by line
    for line in pdb_lines:
        if line.startswith('MODEL'):
            # Extract model number
            try:
                current_model = int(line.split()[1])
                current_lines = [line]  # Start with MODEL line
            except:
                # If model number can't be parsed, use a default
                if current_model is None:
                    current_model = 1
                else:
                    current_model += 1
                current_lines = [f"MODEL     {current_model}\n"]
        elif line.startswith('ENDMDL'):
            # End of a model section
            if current_model is not None:
                current_lines.append(line)
                model_lines[current_model] = current_lines
                current_model = None
        else:
            # Add line to current model
            if current_model is not None:
                current_lines.append(line)
    
    # If the last model doesn't have ENDMDL, add it anyway
    if current_model is not None and current_lines:
        model_lines[current_model] = current_lines
        if not current_lines[-1].startswith('ENDMDL'):
            model_lines[current_model].append("ENDMDL\n")
    
    # Write each model to a separate file
    for model_num, lines in model_lines.items():
        # Construct the output file name using the name schema
        output_file_name = name_schema.format(model_num=model_num)
        output_file = os.path.join(output_folder, f"{output_file_name}.pdb")
        output_files.append(output_file)
        
        # Ensure the output directory exists
        os.makedirs(output_folder, exist_ok=True)
        
        with open(output_file, 'w') as f:
            f.write(f"REMARK   Generated from {input_pdb}, model {model_num}\n")
            for line in lines:
                f.write(line)
            if not lines[-1].startswith('END'):
                f.write("END\n")
    
    print(f"Split {len(output_files)} models from {input_pdb}")
    for file in output_files:
        print(f"  Created: {file}")
    
    return output_files


In [5]:
# Split all models in the PDB file
#output_files = split_pdb_by_model(r"../../data/raw/docking/Flare_prep-flare_dock/BCS-CN76566_T1_D_bcsin_235_pyr1-hab1_P.pdb")

In [ ]:
name="BCS-DR80164"

#nothing to change here except its a tautomer then suffix editing is needed
file_to_split = f"../../data/raw/docking/Results_Flare_Flare/{name}.pdb"
output_dir = f"../../data/raw/docking/Results_Flare_Flare/{name}"
file_name = name + "_S_{model_num}"
output_files = split_pdb_by_model(file_to_split, output_dir, file_name)

